# 工具的应用案例

## 案例1、使用args_schema
arg_schema给出明确的参数信息

In [24]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os


# 1、提供大模型
load_dotenv(override=True)
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")
DASHSCOPE_BASE_URL = os.getenv("DASHSCOPE_BASE_URL")

model = init_chat_model(
    model="qwen3.7-plus",
    model_provider="openai",
    # temperature=1.5,
    api_key=DASHSCOPE_API_KEY,
    base_url=DASHSCOPE_BASE_URL,
    # max_tokens=10,
)

In [25]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from pydantic import BaseModel, Field
from langchain_core.tools.convert import tool
from rich import print as rprint

class WeatherSchema(BaseModel):
    city : str = Field(
        description="具体的城市"
    )
    if_forecast : bool = Field(
        default=True,
        description="是否包含明日天气"
    )

# 自定义工具
@tool(name_or_callable="get_weather_and_forecast", description="查询当日天气，可以包含明天的天气预报",        args_schema=WeatherSchema)
def get_weather(city : str, if_forecast : bool):
    """"""
    res = f"{city}今天天气不错"
    if if_forecast:
        res += "\n明天下雨"
    return res

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather_and_forecast',
        'description': '查询当日天气，可以包含明天的天气预报',
        'parameters': {
            'properties': {
                'city': {'description': '具体的城市', 'type': 'string'},
                'if_forecast': {'default': True, 'description': '是否包含明日天气', 'type': 'boolean'}
            },
            'required': ['city'],
            'type': 'object'
        }
    }
}

In [19]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

# 1、将工具绑定到模型上
model_with_tools = model.bind_tools([get_weather])

# 2、维护一个消息列表
messages = [
    HumanMessage("今天仲恺的天气怎么样？明天呢？")
]

# 3、调用模型 -> AIMessage
response = model_with_tools.invoke(messages)

messages.append(response)

# 4、获取响应中的tool_calls字段信息
tool_calls = response.tool_calls

for tool_call in tool_calls:
    if tool_call["name"] == "get_weather_and_forecast":
        # 5、调用工具（因为大模型不能直接调用工具，所有此时我们主动让工具调用执行）
        # 调用完，返回ToolMessage的实例
        tool_message = get_weather.invoke(tool_call)
        messages.append(tool_message)

# 6、调用模型 -> AIMessage
final_response = model.invoke(messages)

# 7、添加到消息列表中
messages.append(final_response)

# 8、遍历消息列表
for message in messages:
    message.pretty_print()

================================ Human Message =================================

今天仲恺的天气怎么样？明天呢？
================================== Ai Message ==================================
Tool Calls:
  get_weather_and_forecast (call_6fd7e55bc0634ebcb4d75a38)
 Call ID: call_6fd7e55bc0634ebcb4d75a38
  Args:
    city: 仲恺
    if_forecast: True
================================= Tool Message =================================
Name: get_weather_and_forecast

仲恺今天天气不错
明天下雨
================================== Ai Message ==================================

今天仲恺的天气不错，适合外出活动。不过明天会下雨，出门记得带好雨具哦！


## 案例2、撰写docstring
可以在docstring中撰写参数的描述信息，此时参数默认值和类型都要通过函数签名传递。

In [20]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os


# 1、提供大模型
load_dotenv(override=True)
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")
DASHSCOPE_BASE_URL = os.getenv("DASHSCOPE_BASE_URL")

model = init_chat_model(
    model="qwen3.7-plus",
    model_provider="openai",
    # temperature=1.5,
    api_key=DASHSCOPE_API_KEY,
    base_url=DASHSCOPE_BASE_URL,
    # max_tokens=10,
)

In [21]:
from langchain_core.utils.function_calling import convert_to_openai_tool
# from pydantic import BaseModel, Field
from langchain_core.tools.convert import tool
from rich import print as rprint



# 自定义工具
@tool("get_weather_and_forecast", parse_docstring=True)
def get_weather(city : str = "beijing", if_forecast : bool = False):
    """
    查询当日的天气，可以包含明天的天气预报

    Args:
        city : 城市名称
        if_forecast : 是否包含明天天气
    """
    res = f"{city}今天天气不错"
    if if_forecast:
        res += "\n明天下雨"
    return res

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather_and_forecast',
        'description': '查询当日的天气，可以包含明天的天气预报',
        'parameters': {
            'properties': {
                'city': {'default': 'beijing', 'description': '城市名称', 'type': 'string'},
                'if_forecast': {'default': False, 'description': '是否包含明天天气', 'type': 'boolean'}
            },
            'type': 'object'
        }
    }
}

In [22]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

# 1、将工具绑定到模型上
model_with_tools = model.bind_tools([get_weather])

# 2、维护一个消息列表
messages = [
    HumanMessage("今天仲恺的天气怎么样？明天呢？")
]

# 3、调用模型 -> AIMessage
response = model_with_tools.invoke(messages)

messages.append(response)

# 4、获取响应中的tool_calls字段信息
tool_calls = response.tool_calls

for tool_call in tool_calls:
    if tool_call["name"] == "get_weather_and_forecast":
        # 5、调用工具（因为大模型不能直接调用工具，所有此时我们主动让工具调用执行）
        # 调用完，返回ToolMessage的实例
        tool_message = get_weather.invoke(tool_call)
        messages.append(tool_message)

# 6、调用模型 -> AIMessage
final_response = model.invoke(messages)

# 7、添加到消息列表中
messages.append(final_response)

# 8、遍历消息列表
for message in messages:
    message.pretty_print()

================================ Human Message =================================

今天仲恺的天气怎么样？明天呢？
================================== Ai Message ==================================
Tool Calls:
  get_weather_and_forecast (call_06ecc070fd2c4ec3bbb1c3a3)
 Call ID: call_06ecc070fd2c4ec3bbb1c3a3
  Args:
    city: 仲恺
    if_forecast: True
================================= Tool Message =================================
Name: get_weather_and_forecast

仲恺今天天气不错
明天下雨
================================== Ai Message ==================================

今天仲恺的天气不错，适合外出。不过明天下雨，如果您明天有出行计划，记得提前准备好雨具哦！


## 案例3、多工具调用

In [23]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os


# 1、提供大模型
load_dotenv(override=True)
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")
DASHSCOPE_BASE_URL = os.getenv("DASHSCOPE_BASE_URL")

model = init_chat_model(
    model="qwen3.7-plus",
    model_provider="openai",
    # temperature=1.5,
    api_key=DASHSCOPE_API_KEY,
    base_url=DASHSCOPE_BASE_URL,
    # max_tokens=10,
)

In [29]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
# from rich import print as rprint
# from langchain_core.utils.function_calling import convert_to_openai_tool


# 1.定义工具
# 定义股票查询工具
@tool(parse_docstring=True)
def get_stock_price(company: str, timeframe: str = "today") -> str:
    """
    获取指定公司的股票价格信息

    Args:
        company: 公司名称（如：苹果公司, 微软公司, 谷歌公司）
        timeframe: 时间范围（today-今日, week-本周, month-本月）
    """
    # 模拟股票数据
    mock_data = {
        "苹果公司": {"today": 185.20, "week": 183.50, "month": 180.75},
        "微软公司": {"today": 415.86, "week": 412.30, "month": 405.42},
        "谷歌公司": {"today": 15.42, "week": 15.20, "month": 14.85}
    }
    if company in mock_data:
        price = mock_data[company].get(timeframe, "未知时间范围")
        return f"{company} {timeframe}价格: {price}美元"
    else:
        return f"未找到股票代码 {company} 的数据"

# 定义新闻搜索工具
@tool(parse_docstring=True)
def search_news(company: str) -> str:
    """
    搜索指定公司的财经新闻

    Args:
        company: 公司名称

    Returns:
        公司的财经新闻，每条新闻占一行
    """
    # 模拟新闻数据
    mock_news = {
        "苹果公司": [
            "苹果发布新款iPhone，股价上涨3%",
            "苹果与欧盟达成反垄断和解协议",
            "苹果将在印度扩大生产规模"
        ],
        "微软公司": [
            "微软Azure云业务季度增长超预期",
            "微软完成对Nuance的收购",
            "微软推出新一代AI助手Copilot"
        ],
        "谷歌公司": [
            "谷歌发布新AI模型，性能提升20%",
            "谷歌与OpenAI合作，开发新的AI助手",
            "谷歌在欧洲展开AI研究项目"
        ]
    }
    news_list = mock_news.get(company, [f"未找到{company}的相关新闻"])
    return "\n".join(news_list)

# rprint(convert_to_openai_tool(search_news))

# 2.初始化模型并绑定工具
tools = [get_stock_price, search_news]
model_with_tools = model.bind_tools(tools)
message_list = []
human_message = HumanMessage(content="苹果公司今天的股价是多少？最近有什么新闻？")
# human_message = HumanMessage(content="比较一下微软和苹果的股价")
# human_message = HumanMessage(content="腾讯最近有什么重大新闻？")
# human_message = HumanMessage(content="海水为什么是咸的？")
message_list.append(human_message)

# 3.工具调用循环
while True:
    response = model_with_tools.invoke(message_list)

    # rprint(response)
    #
    # break
    message_list.append(response)
    # 如果模型不需要调用工具，直接退出循环
    if not response.tool_calls:
        print("没有工具调用，直接返回答案")
        break
    # 如果有调用工具，处理工具调用响应
    for tool_call in response.tool_calls:
        if tool_call["name"] == "get_stock_price":
            stock_result = get_stock_price.invoke(tool_call)
            print("stock_result", stock_result)
            message_list.append(stock_result)
        if tool_call["name"] == "search_news":
            news_result = search_news.invoke(tool_call)
            print("news_result", news_result)
            message_list.append(news_result)

for msg in message_list:
    msg.pretty_print()

stock_result content='苹果公司 today价格: 185.2美元' name='get_stock_price' tool_call_id='call_56d531fc14714e50bb1c9cd4'
news_result content='苹果发布新款iPhone，股价上涨3%\n苹果与欧盟达成反垄断和解协议\n苹果将在印度扩大生产规模' name='search_news' tool_call_id='call_c7946b6b52264aad90a6ed9c'
没有工具调用，直接返回答案
================================ Human Message =================================

苹果公司今天的股价是多少？最近有什么新闻？
================================== Ai Message ==================================
Tool Calls:
  get_stock_price (call_56d531fc14714e50bb1c9cd4)
 Call ID: call_56d531fc14714e50bb1c9cd4
  Args:
    company: 苹果公司
    timeframe: today
  search_news (call_c7946b6b52264aad90a6ed9c)
 Call ID: call_c7946b6b52264aad90a6ed9c
  Args:
    company: 苹果公司
================================= Tool Message =================================
Name: get_stock_price

苹果公司 today价格: 185.2美元
================================= Tool Message =================================
Name: search_news

苹果发布新款iPhone，股价上涨3%
苹果与欧盟达成反垄断和解协议
苹果将在印度扩大生产规模
=================

## 案例4、多工具调用

In [31]:
from langchain.tools import tool
from langchain_core.messages import HumanMessage

@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """
    获取当日天气

    Args:
        city: 城市名称
    """
    return f'{city}当天晴朗'

@tool(parse_docstring=True)
def get_news() -> str:
    """
    获取当日新闻
    """
    return "近期，受全球储蓄芯片短缺等多重因素影响，多地回收商称废旧手机回收市场迎来“火热潮”，回收价格普遍上涨，旧手机成“香饽饽”。"

model_with_tools = model.bind_tools([get_weather, get_news])
messages = [
    HumanMessage(content="今天杭州天气如何？今天新闻是什么？别瞎编")
]

# 第一步：模型生成调用工具的指令
response = model_with_tools.invoke(messages)
messages.append(response)

# 第二步：遍历执行工具
for tool_call in response.tool_calls:
    if tool_call["name"] == "get_weather":
        tool_msg = get_weather.invoke(tool_call)
        print(tool_msg)
        messages.append(tool_msg)
    elif tool_call["name"] == "get_news":
        tool_msg = get_news.invoke(tool_call)
        print(tool_msg)
        messages.append(tool_msg)
    else:
        raise Exception("不存在的工具")

# 第三步：将工具执行结果交给模型生成最终回答
final_response = model.invoke(messages)
messages.append(final_response)

# 打印全部消息
for msg in messages:
    msg.pretty_print()

content='杭州当天晴朗' name='get_weather' tool_call_id='call_0088d6a99e0240ee9aa5fa89'
content='近期，受全球储蓄芯片短缺等多重因素影响，多地回收商称废旧手机回收市场迎来“火热潮”，回收价格普遍上涨，旧手机成“香饽饽”。' name='get_news' tool_call_id='call_0e7e4e2eddd34a128dcc5b15'
================================ Human Message =================================

今天杭州天气如何？今天新闻是什么？别瞎编
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_0088d6a99e0240ee9aa5fa89)
 Call ID: call_0088d6a99e0240ee9aa5fa89
  Args:
    city: 杭州
  get_news (call_0e7e4e2eddd34a128dcc5b15)
 Call ID: call_0e7e4e2eddd34a128dcc5b15
  Args:
================================= Tool Message =================================
Name: get_weather

杭州当天晴朗
================================= Tool Message =================================
Name: get_news

近期，受全球储蓄芯片短缺等多重因素影响，多地回收商称废旧手机回收市场迎来“火热潮”，回收价格普遍上涨，旧手机成“香饽饽”。
================================== Ai Message ==================================

今天杭州的天气是**晴朗**。

关于今天的新闻：
近期，受全球储蓄芯片短缺等多重因素影